# Implementação do Modelo - Fase 2

Reogarnização dos arquivos do projeto para que reflitam as fases aplicadas na pesquisa. Conforme Guia_Fase2_Passo2_Estrutura_Modulos.cd

In [1]:
%load_ext autoreload
%autoreload 2


import random

from philder import infra
from philder import database_consolidation as consolidacao
from philder import data_processing as processamento
from philder import database_balance as balanceamento
from philder import performance_assessment as avaliacao


# --- Preparação ---
infra.criar_estrutura_diretorios()

# --- Módulo A: consolidação da base ---
#brutos_legitimos = consolidacao.importar_corpus_spamassassin("dados_philder/spamassassin")
#brutos_phishing = consolidacao.importar_corpus_nazario("dados_philder/Nazario.csv")
#df = consolidacao.consolidar_dataset(brutos_legitimos, brutos_phishing)

# --- Módulo B: processamento de texto ---
# df = processamento.pipeline_processamento(df)

# --- Módulo C: balanceamento + treino + teste ---
# df_balanceado = balanceamento.aplicar_undersampling(df)
# df_treino, df_teste = balanceamento.dividir_treino_teste(df_balanceado)
# tokenizer, modelo_tb = balanceamento.carregar_tokenizer_e_modelo()
# dataset_treino, dataset_teste = balanceamento.tokenizar_dataset(df_treino, df_teste, tokenizer)
# trainer, tempo_treino = balanceamento.treinar_modelo(modelo_tb, tokenizer, dataset_treino, dataset_teste)
# predicoes, tempo_teste = balanceamento.testar_modelo(trainer, dataset_teste)
# balanceamento.salvar_modelo(trainer, tokenizer)

# --- Módulo D: avaliação de desempenho ---
# metricas = avaliacao.calcular_metricas_primarias(df_teste["class_label"].tolist(), predicoes)
# eficiencia = avaliacao.calcular_metricas_eficiencia(metricas["accuracy"], tempo_treino, tempo_teste, len(df_teste))
# avaliacao.gerar_curva_perda(trainer.state.log_history)
# avaliacao.salvar_resultados({**metricas, **eficiencia})


Estrutura de diretórios criada/confirmada:
  - dados_philder/
  - modelo/  (com modelo\checkpoints/)
  - resultados/


# A) DATABASE CONSOLIDATION

<DESENVOLVER>

In [2]:
# --- Módulo A: consolidação da base ---
brutos_legitimos = consolidacao.importar_corpus_spamassassin("dados_philder/spamassassin")
brutos_phishing = consolidacao.importar_corpus_nazario("dados_philder/Nazario.csv")

# 19/09/2026 -  Ajuste da base para refletir a quantidade de amostras do artigo original, especifcamente
# da base do phishtank
# Nazario tem mais e-mails de phishing do que o PhishTank usado no artigo (501) —
# reduz por amostragem aleatória (sem reposição) para equiparar a mesma quantidade.
# SpamAssassin já está na mesma contagem do artigo (2.501), não precisa de ajuste.
random.seed(42)
brutos_phishing = random.sample(brutos_phishing, 501)

df = consolidacao.consolidar_dataset(brutos_legitimos, brutos_phishing)

Total de e-mails legítimos carregados: 2501
Total de e-mails de phishing carregados: 1565 (coluna: 'body')
Dataset consolidado: 3002 registros — legítimos: 2501, phishing: 501


# B) DATA PROCESSING
<DISCORRER MAIS>

In [5]:
# --- Módulo B: processamento de texto ---
df = processamento.pipeline_processamento(df)

## Debug do Resultado e análise do dataset de phishing, quanto a amostras duplicadas

print("Duplicatas exatas no corpo do texto:", df["body"].duplicated().sum())
print("Duplicatas após processamento:", df["body_processado"].duplicated().sum())

df = df.drop_duplicates(subset="body_processado").reset_index(drop=True)
print(f"Dataset após remover duplicatas: {len(df)} registros")

# Remoção de amostras duplicas, para evitar "vazamento de dados no treinamento do modelo"


Pré-processamento concluído: 3002 registros (16 ficaram vazios)
Duplicatas exatas no corpo do texto: 30
Duplicatas após processamento: 76
Dataset após remover duplicatas: 2926 registros


# C) DATABASE BALANCE

In [6]:
# --- Módulo C: balanceamento + treino + teste ---
df_balanceado = balanceamento.aplicar_undersampling(df)
df_treino, df_teste = balanceamento.dividir_treino_teste(df_balanceado)

### Debug e análise do resultado. Checando deduplicação:  

## sobreposicao = set(df_treino["body_processado"]) & set(df_teste["body_processado"])
## print(f"E-mails processados idênticos entre treino e teste: {len(sobreposicao)}")
## raise SystemExit("Checagem de duplicatas concluída — parando aqui antes do treino.")

tokenizer, modelo_tb = balanceamento.carregar_tokenizer_e_modelo()
dataset_treino, dataset_teste = balanceamento.tokenizar_dataset(df_treino, df_teste, tokenizer)
trainer, tempo_treino = balanceamento.treinar_modelo(modelo_tb, tokenizer, dataset_treino, dataset_teste)
predicoes, tempo_teste = balanceamento.testar_modelo(trainer, dataset_teste)
balanceamento.salvar_modelo(trainer, tokenizer)

Balanceamento aplicado: {0: 481, 1: 481}
Treino: {1: 337, 0: 336}
Teste: {0: 145, 1: 144}


Loading weights:   0%|          | 0/71 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: huawei-noah/TinyBERT_General_4L_312D
Key                                        | Status     | 
-------------------------------------------+------------+-
fit_denses.{0, 1, 2, 3, 4}.weight          | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
fit_denses.{0, 1, 2, 3, 4}.bias            | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/arch

Map:   0%|          | 0/673 [00:00<?, ? examples/s]

Map:   0%|          | 0/289 [00:00<?, ? examples/s]

Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.191309,0.166330,0.944637,0.942029
2,0.122876,0.130457,0.968858,0.967972
3,0.004543,0.143130,0.968858,0.967972
4,0.012511,0.134202,0.975779,0.975265
5,0.075473,0.113357,0.979239,0.979167
6,0.001349,0.124025,0.979239,0.979167
7,0.001128,0.110222,0.979239,0.979167


... (7 outputs left out)

Tempo de treino: 298.1s


Tempo total de inferência no teste: 2.07s


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Modelo salvo em: modelo/


# D) PERFORMANCE ASSESSMENT
## Alterações identificadas:

- [x] Incluir na tabela de métricas o Tempo de treino e o tempo de teste e tempo de detecção
- [ ] Equacionar o número de épocas com o do artigo (10 épocas)
- [ ] Identificar padrão de unidades usada no artigo para avaliar ajuste e necessidade de conversão de dados, pois os resultados são bem diferentes. Considerar também o hardware utilizado
- [ ]  Finalizado ajustes rodar notebook no google colabs e comparar os resultados IES, TES, RTDE
- [x] Executar um treinamento com o dataset atual, simulando a quantidade de amostras do artigo original

## Resultados do artigo original:

### Tabela 1 — Métricas de Desempenho Preditivo (PHILDER = TinyBERT (2), execução com undersampling)

| Modelo | Acc. | Prec. | Rec. | F1 | T. Treino (S)| T. Teste (s) | Tempo de Detecção (ms) |
|---|---|---|---|---|---|---|---|
| TinyBERT (2) | 0,9927 | 0,9896 | 0,9960 | 0,9927 | 111,83 | 4,48 | 14,8841 |

### Tabela 2 — Métricas de Eficiência Computacional (PHILDER = TinyBERT + undersampling, valores brutos dos gráficos)
**Valores não normalizados**
| Métrica | Valor bruto |
|---|---|
| TES | 0,0389 |
| IES | 1,0788 |
| RTDE | 324,7750 |

## Resultados deste Trabalho Fase2

### Tabela 1 — Métricas de Desempenho Preditivo

**Data Medição: 17/09/2026**

**Base Nazario 1501 amostras**

| Modelo | Acc. | Prec. | Rec. | F1 |
|---|---|---|---|---|
| philder |0,9936 | 0,9957 | 0,9915 | 0,9936 |

**Data Medição: 19/09/2026**

**Base Nazario 501 amostras**
| Modelo | Acc. | Prec. | Rec. | F1 | T. Treino (S)| T. Teste (s) | Tempo de Detecção (ms) |
|---|---|---|---|---|---|---|---|
| philder |0,9867 | 1,0 | 0,9733 | 0,9865 | 255.8025 | 4.5329 | 15.0595 |

**Data Medição: 21/09/2026**

**Base Nazario 501 amostras**

**10 épocas, + parametros p/ evitar overfiting**

| Modelo | Acc. | Prec. | Rec. | F1 | T. Treino (S)| T. Teste (s) | Tempo de Detecção (ms) |
|---|---|---|---|---|---|---|---|
| philder |0,9933 | 1,0 | 0,9867 | 0,9933 | 205,5063 | 2,1591 | 7,1734 |


### Tabela 2 — Métricas de Eficiência Computacional

**Data Medição: 17/09/2026**

**Base Nazario 1501 amostras**

| Métrica | Valor bruto |
|---|---|
| TES | 0,00268 |
| IES | 0,1474 |
| RTDE | 138,3831 |

**Data Medição: 19/09/2026**

**Base Nazario 501 amostras**

| Métrica | Valor bruto |
|---|---|
| TES | 0,0039 |
| IES | 0,2177 |
| RTDE | 65,505 |

**Data Medição: 19/09/2026**

**Base Nazario 501 amostras**

**10 épocas, + parametros p/  evitar overfiting**

| Métrica | Valor bruto |
|---|---|
| TES | 0,0048 |
| IES | 0,4601 |
| RTDE | 138/,4772 |

## Análise Crítica da Avaliação dos Resultados

- Até o momento os resultados de Eficiência são da ordem de 10x piores que o do artigo, indicando que as etapas de treino e testes foram executadoas em um ambiente de poder computacional elevado. O fator mais relevante para a discrepância nos valores foi o tempo. Avaliar este ponto com mais atenção e garantir que as métricas estão realizando o calculo da forma adequada,
- Verificado que, após a equalização da base com o artigo, foi identificado que algumas amostras identicas no dataset treinado incfluenciaram na precisão do modelo (100%). Para contornar a situação foi realizado um tratamento no dataset para a normalização dos duplicados `df.drop_duplicates` e realizada nova medição com este dataset normalizado. 
- Aparentemente os valores estão normalizados (0,1). Avaliar qual técnica foi utilizada (normalização?)

In [7]:
# --- Módulo D: avaliação de desempenho ---
metricas = avaliacao.calcular_metricas_primarias(df_teste["class_label"].tolist(), predicoes)
eficiencia = avaliacao.calcular_metricas_eficiencia(metricas["accuracy"], tempo_treino, tempo_teste, len(df_teste))
avaliacao.gerar_curva_perda(trainer.state.log_history)
avaliacao.salvar_resultados({**metricas, **eficiencia})

Curva de perda salva em: resultados/curva_perda.png
Resumo salvo em: resultados/resultado_final.csv
